# 04 - Task A: Binary Screening Classification (Normal vs ROP)
### UROP: Improving Cross-Dataset Generalization and Severe-Stage Detection in ROP
This notebook implements **Task A** experiments studying domain shift and model performance:
- **Experiment A1**: Pooled patient-level 5-fold cross-validation across ROP-VL and Ostrava.
- **Architectures Compared**: ResNet-50 (primary baseline), EfficientNet-B0 (lightweight), ConvNeXt-Tiny (modern CNN).
- **Training Runs Compared**:
  * Run 1: Standard Cross-Entropy
  * Run 2: Class-Weighted Cross-Entropy
  * Run 3: Focal Loss
  * Run 4: Stronger Domain-Oriented Augmentation
  * Run 5: Best Combination


In [2]:
import os, sys
if "COLAB_GPU" in os.environ or os.path.exists("/content"):
    sys.path.append("/content/drive/MyDrive/ROP_UROP")
    sys.path.append("/content/ROP_UROP")
else:
    sys.path.append(r"d:/UROP")

import torch
import pandas as pd
from src.models import get_model, count_parameters
from src.dataset import build_dataloaders
from src.losses import get_loss_function
from src.training import Trainer, set_seed
from configs import config

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [3]:
# Load Task A dataset
taskA_path = config.MANIFESTS_DIR / "taskA_manifest.csv"
taskA_df = pd.read_csv(taskA_path)
print(f"Total Task A images: {len(taskA_df)}")

# Model Parameter Comparison
for name in ["resnet50", "efficientnet_b0", "convnext_tiny"]:
    m = get_model(name, num_classes=2, pretrained=False)
    tot, trn = count_parameters(m)
    print(f"Backbone: {name:<16} | Total Params: {tot/1e6:.2f}M | Trainable: {trn/1e6:.2f}M")


Total Task A images: 6145
Backbone: resnet50         | Total Params: 23.51M | Trainable: 23.51M
Backbone: efficientnet_b0  | Total Params: 4.01M | Trainable: 4.01M
Backbone: convnext_tiny    | Total Params: 27.82M | Trainable: 27.82M


In [4]:
# Run 1: Baseline ResNet-50 with Standard Cross-Entropy on Fold 0
train_loader, val_loader = build_dataloaders(taskA_df, val_fold=0, batch_size=config.BATCH_SIZE, task="binary")

model = get_model("resnet50", num_classes=2, pretrained=True)
criterion = get_loss_function("cross_entropy", device=device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    experiment_name="taskA_resnet50_ce_fold0",
    task="binary",
    patience=5
)

# In Colab with GPU, train for 15-20 epochs; on CPU run 1-2 demo epochs
epochs = 15 if torch.cuda.is_available() else 1
history = trainer.train(train_loader, val_loader, epochs=epochs)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\nayan/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:05<00:00, 17.6MB/s]
d:\UROP\src\training.py:71: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.use_amp)



--- Starting Experiment: taskA_resnet50_ce_fold0 on cpu ---
Epochs: 1 | Batch size: 16 | Task: binary


d:\UROP\src\training.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.use_amp):
d:\UROP\src\training.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.use_amp):


Epoch [01/01] Train Loss: 0.3559 | Val Loss: 0.3218 | Val Acc: 0.8570 | AUROC: 0.9350 | F1: 0.8413 | (2769.1s)
  --> Saved new best checkpoint (Metric: 0.9350)

Training completed. Best monitored metric: 0.9350


In [6]:
# Evaluate Best Model & Compute Clinical Metrics with 95% Bootstrap CIs
from src.evaluation import compute_binary_metrics, compute_bootstrap_ci
from src.visualization import plot_learning_curves, plot_roc_pr_curves, plot_confusion_matrix

trainer.load_best_model()
val_loss, metrics = trainer.evaluate(val_loader)

print("Validation Metrics:")
for k, v in metrics.items():
    if k != "confusion_matrix":
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Plot learning curves
plot_learning_curves(history, config.FIGURES_DIR / "taskA_learning_curves.png", title="Task A ResNet-50")
plot_confusion_matrix(metrics["confusion_matrix"], ["Normal", "ROP"], config.FIGURES_DIR / "taskA_confusion_matrix.png")


[Loaded] Best weights loaded from D:\UROP\checkpoints\taskA_resnet50_ce_fold0_best.pt


d:\UROP\src\training.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.use_amp):


Validation Metrics:
  accuracy: 0.8570
  sensitivity: 0.8243
  recall: 0.8243
  specificity: 0.8848
  precision: 0.8590
  f1_score: 0.8413
  auroc: 0.9350
  auprc: 0.9322
  tp: 591
  fp: 97
  tn: 745
  fn: 126
  total_images: 1559
[Plot Saved] D:\UROP\figures\taskA_learning_curves.png
[Plot Saved] D:\UROP\figures\taskA_confusion_matrix.png
